# STAGE 5.4 — ROI QUALITY VALIDATION

In [2]:
# ============================================================
# CISLR / OPENVINO RESEARCH
# STAGE 5.4 — ROI QUALITY VALIDATION
# ============================================================
#
# PURPOSE
# -------
# Validate the quality of the Hand-ROI representation produced
# during Stage 5.3.
#
# IMPORTANT:
#   Stage 5.4 DOES NOT rerun OpenCV detection.
#   Stage 5.4 DOES NOT read raw videos.
#   Stage 5.4 DOES NOT regenerate ROI images.
#
# It audits the frozen Stage 5.3 outputs using:
#
#   - detection_valid
#   - fallback_type
#   - selected_num_hands
#   - confidence_proxy
#   - bbox_area_fraction
#   - roi_saved
#   - latency_ms
#
# Main validation dimensions:
#
#   1. Detection success
#   2. Missing-hand / fallback behavior
#   3. One-hand / two-hand distribution
#   4. ROI size
#   5. Confidence distribution
#   6. Temporal consistency
#   7. Video-level ROI quality
#   8. Frame-level ROI quality
#   9. Quality threshold analysis
#  10. Final Stage 5.4 acceptance decision
#
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import matplotlib.pyplot as plt
from datetime import datetime

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi"
)

EXTRACT_DIR = ROOT / "roi_extraction"

FRAME_AUDIT = (
    EXTRACT_DIR /
    "stage5_3_frame_level_extraction.csv"
)

VIDEO_AUDIT = (
    EXTRACT_DIR /
    "stage5_3_video_level_summary.csv"
)

VALID_MANIFEST = (
    ROOT /
    "analysis_valid_subset" /
    "stage5_3_analysis_valid_uid_manifest.csv"
)

OUTPUT_DIR = (
    ROOT /
    "roi_quality_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PLOTS_DIR = (
    OUTPUT_DIR /
    "plots"
)

PLOTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 100)
print("STAGE 5.4 — ROI QUALITY VALIDATION")
print("=" * 100)

# ------------------------------------------------------------
# 2. VERIFY INPUTS
# ------------------------------------------------------------

required_inputs = {
    "Stage 5.3 frame audit": FRAME_AUDIT,
    "Stage 5.3 video audit": VIDEO_AUDIT,
    "Frozen analysis-valid UID manifest": VALID_MANIFEST,
}

print("\nInput verification:\n")

missing = []

for description, path in required_inputs.items():

    exists = path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{description}"
    )

    print(f"    {path}")

    if not exists:
        missing.append(path)

if missing:

    raise FileNotFoundError(
        "\nMissing required Stage 5.4 inputs:\n"
        +
        "\n".join(str(p) for p in missing)
    )

print("\nAll Stage 5.4 inputs found.")

# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

frame_df = pd.read_csv(FRAME_AUDIT)

video_df = pd.read_csv(VIDEO_AUDIT)

valid_manifest = pd.read_csv(VALID_MANIFEST)

print("\nLoaded shapes:")

print(
    "Frame audit :",
    frame_df.shape
)

print(
    "Video audit :",
    video_df.shape
)

print(
    "Valid UID manifest :",
    valid_manifest.shape
)

# ------------------------------------------------------------
# 4. FREEZE ANALYSIS UID SET
# ------------------------------------------------------------

if "uid" not in valid_manifest.columns:

    raise KeyError(
        "Frozen UID manifest does not contain 'uid'."
    )

valid_uids = set(
    valid_manifest["uid"]
    .astype(str)
)

frame_df["uid"] = (
    frame_df["uid"]
    .astype(str)
)

video_df["uid"] = (
    video_df["uid"]
    .astype(str)
)

print("\nFrozen analysis UID count:")
print(len(valid_uids))

# ------------------------------------------------------------
# 5. FILTER TO ANALYSIS-VALID VIDEOS
# ------------------------------------------------------------

analysis_frames = frame_df[
    frame_df["uid"].isin(valid_uids)
].copy()

analysis_videos = video_df[
    video_df["uid"].isin(valid_uids)
].copy()

print("\nMatched analysis population:")

print(
    "Videos :",
    analysis_videos["uid"].nunique()
)

print(
    "Frames :",
    len(analysis_frames)
)

EXPECTED_VIDEOS = 6628
EXPECTED_FRAMES = EXPECTED_VIDEOS * 8

if analysis_videos["uid"].nunique() != EXPECTED_VIDEOS:

    raise ValueError(
        "Unexpected number of analysis-valid videos. "
        f"Expected {EXPECTED_VIDEOS}, got "
        f"{analysis_videos['uid'].nunique()}."
    )

if len(analysis_frames) != EXPECTED_FRAMES:

    raise ValueError(
        "Unexpected number of analysis-valid frames. "
        f"Expected {EXPECTED_FRAMES}, got "
        f"{len(analysis_frames)}."
    )

print(
    f"\n✓ Confirmed {EXPECTED_VIDEOS} videos × 8 frames "
    f"= {EXPECTED_FRAMES} frame records."
)

# ------------------------------------------------------------
# 6. REQUIRED COLUMN VALIDATION
# ------------------------------------------------------------

required_frame_columns = [

    "uid",
    "sample_number",
    "frame_index",
    "frame_read",
    "decoder",
    "detector",
    "detection_valid",
    "fallback_type",
    "selected_num_hands",
    "confidence_proxy",
    "bbox_area_fraction",
    "roi_saved",
    "latency_ms",
    "error",
]

missing_columns = [
    c
    for c in required_frame_columns
    if c not in analysis_frames.columns
]

if missing_columns:

    raise KeyError(
        "Missing required frame-audit columns:\n"
        +
        "\n".join(missing_columns)
    )

print(
    "\n✓ Required frame-level quality columns present."
)

# ------------------------------------------------------------
# 7. NORMALIZE BOOLEAN / NUMERIC COLUMNS
# ------------------------------------------------------------

def normalize_bool(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        })
        .fillna(False)
    )


analysis_frames["frame_read_bool"] = normalize_bool(
    analysis_frames["frame_read"]
)

analysis_frames["detection_valid_bool"] = normalize_bool(
    analysis_frames["detection_valid"]
)

analysis_frames["roi_saved_bool"] = normalize_bool(
    analysis_frames["roi_saved"]
)

analysis_frames["selected_num_hands"] = pd.to_numeric(
    analysis_frames["selected_num_hands"],
    errors="coerce"
)

analysis_frames["confidence_proxy"] = pd.to_numeric(
    analysis_frames["confidence_proxy"],
    errors="coerce"
)

analysis_frames["bbox_area_fraction"] = pd.to_numeric(
    analysis_frames["bbox_area_fraction"],
    errors="coerce"
)

analysis_frames["latency_ms"] = pd.to_numeric(
    analysis_frames["latency_ms"],
    errors="coerce"
)

analysis_frames["sample_number"] = pd.to_numeric(
    analysis_frames["sample_number"],
    errors="coerce"
)

analysis_frames["frame_index"] = pd.to_numeric(
    analysis_frames["frame_index"],
    errors="coerce"
)

# ------------------------------------------------------------
# 8. NORMALIZE FALLBACK TYPE
# ------------------------------------------------------------

analysis_frames["fallback_type"] = (
    analysis_frames["fallback_type"]
    .fillna("none")
    .astype(str)
    .str.strip()
)

analysis_frames["fallback_type_lower"] = (
    analysis_frames["fallback_type"]
    .str.lower()
)

# ------------------------------------------------------------
# 9. IDENTIFY DIRECT / FALLBACK CATEGORIES
# ------------------------------------------------------------

analysis_frames["is_direct_detection"] = (
    analysis_frames["detection_valid_bool"]
    &
    (
        ~analysis_frames["fallback_type_lower"]
        .isin([
            "temporal_fallback",
            "full_frame_fallback",
        ])
    )
)

analysis_frames["is_temporal_fallback"] = (
    analysis_frames["fallback_type_lower"]
    == "temporal_fallback"
)

analysis_frames["is_full_frame_fallback"] = (
    analysis_frames["fallback_type_lower"]
    == "full_frame_fallback"
)

analysis_frames["is_any_fallback"] = (
    analysis_frames["is_temporal_fallback"]
    |
    analysis_frames["is_full_frame_fallback"]
)

# ------------------------------------------------------------
# 10. ONE-HAND / TWO-HAND CATEGORIES
# ------------------------------------------------------------

analysis_frames["is_one_hand"] = (
    analysis_frames["selected_num_hands"] == 1
)

analysis_frames["is_two_hands"] = (
    analysis_frames["selected_num_hands"] >= 2
)

analysis_frames["is_zero_hands"] = (
    analysis_frames["selected_num_hands"] == 0
)

# ------------------------------------------------------------
# 11. ROI AREA VALIDATION
# ------------------------------------------------------------

# The Stage 5.2 detector-selection protocol used:
#
#   minimum ROI area fraction = 0.005
#   maximum ROI area fraction = 0.90
#
# Stage 5.4 also records a softer practical target around 20%.
#
ROI_MIN = 0.005
ROI_MAX = 0.90
ROI_TARGET = 0.20

analysis_frames["roi_area_valid"] = (
    analysis_frames["bbox_area_fraction"]
    .between(
        ROI_MIN,
        ROI_MAX,
        inclusive="both"
    )
)

analysis_frames["roi_area_near_target"] = (
    analysis_frames["bbox_area_fraction"]
    .between(
        0.10,
        0.40,
        inclusive="both"
    )
)

# ------------------------------------------------------------
# 12. FRAME-LEVEL QUALITY STATUS
# ------------------------------------------------------------

analysis_frames["quality_status"] = "invalid"

analysis_frames.loc[
    analysis_frames["roi_saved_bool"],
    "quality_status"
] = "saved"

analysis_frames.loc[
    analysis_frames["is_direct_detection"]
    &
    analysis_frames["roi_saved_bool"],
    "quality_status"
] = "direct_detection"

analysis_frames.loc[
    analysis_frames["is_temporal_fallback"]
    &
    analysis_frames["roi_saved_bool"],
    "quality_status"
] = "temporal_fallback"

analysis_frames.loc[
    analysis_frames["is_full_frame_fallback"]
    &
    analysis_frames["roi_saved_bool"],
    "quality_status"
] = "full_frame_fallback"

# ------------------------------------------------------------
# 13. FRAME-LEVEL SUMMARY
# ------------------------------------------------------------

N_FRAMES = len(analysis_frames)

frame_summary = {

    "analysis_videos":
        int(analysis_frames["uid"].nunique()),

    "analysis_frames":
        int(N_FRAMES),

    "frame_read_success_rate":
        float(
            analysis_frames["frame_read_bool"].mean()
        ),

    "roi_saved_rate":
        float(
            analysis_frames["roi_saved_bool"].mean()
        ),

    "direct_detection_rate":
        float(
            analysis_frames["is_direct_detection"].mean()
        ),

    "temporal_fallback_rate":
        float(
            analysis_frames["is_temporal_fallback"].mean()
        ),

    "full_frame_fallback_rate":
        float(
            analysis_frames["is_full_frame_fallback"].mean()
        ),

    "any_fallback_rate":
        float(
            analysis_frames["is_any_fallback"].mean()
        ),

    "one_hand_rate":
        float(
            analysis_frames["is_one_hand"].mean()
        ),

    "two_hand_rate":
        float(
            analysis_frames["is_two_hands"].mean()
        ),

    "zero_hand_rate":
        float(
            analysis_frames["is_zero_hands"].mean()
        ),

    "roi_area_valid_rate":
        float(
            analysis_frames["roi_area_valid"].mean()
        ),

    "roi_area_near_target_rate":
        float(
            analysis_frames["roi_area_near_target"].mean()
        ),
}

# ------------------------------------------------------------
# 14. CONFIDENCE STATISTICS
# ------------------------------------------------------------

confidence = (
    analysis_frames["confidence_proxy"]
    .dropna()
)

confidence_summary = {

    "count":
        int(len(confidence)),

    "mean":
        float(confidence.mean()),

    "median":
        float(confidence.median()),

    "std":
        float(confidence.std()),

    "min":
        float(confidence.min()),

    "p05":
        float(confidence.quantile(0.05)),

    "p25":
        float(confidence.quantile(0.25)),

    "p75":
        float(confidence.quantile(0.75)),

    "p95":
        float(confidence.quantile(0.95)),

    "max":
        float(confidence.max()),
}

# ------------------------------------------------------------
# 15. ROI AREA STATISTICS
# ------------------------------------------------------------

roi_area = (
    analysis_frames["bbox_area_fraction"]
    .dropna()
)

roi_area_summary = {

    "count":
        int(len(roi_area)),

    "mean":
        float(roi_area.mean()),

    "median":
        float(roi_area.median()),

    "std":
        float(roi_area.std()),

    "min":
        float(roi_area.min()),

    "p05":
        float(roi_area.quantile(0.05)),

    "p25":
        float(roi_area.quantile(0.25)),

    "p75":
        float(roi_area.quantile(0.75)),

    "p95":
        float(roi_area.quantile(0.95)),

    "max":
        float(roi_area.max()),
}

# ------------------------------------------------------------
# 16. LATENCY STATISTICS
# ------------------------------------------------------------

latency = (
    analysis_frames["latency_ms"]
    .dropna()
)

latency_summary = {

    "count":
        int(len(latency)),

    "mean_ms":
        float(latency.mean()),

    "median_ms":
        float(latency.median()),

    "p95_ms":
        float(latency.quantile(0.95)),

    "max_ms":
        float(latency.max()),
}

# ------------------------------------------------------------
# 17. FALLBACK DISTRIBUTION
# ------------------------------------------------------------

fallback_distribution = (
    analysis_frames["fallback_type"]
    .value_counts(dropna=False)
    .rename_axis("fallback_type")
    .reset_index(name="frame_count")
)

fallback_distribution["fraction"] = (
    fallback_distribution["frame_count"]
    / N_FRAMES
)

# ------------------------------------------------------------
# 18. HAND COUNT DISTRIBUTION
# ------------------------------------------------------------

hand_distribution = (
    analysis_frames["selected_num_hands"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("num_hands")
    .reset_index(name="frame_count")
)

hand_distribution["fraction"] = (
    hand_distribution["frame_count"]
    / N_FRAMES
)

# ------------------------------------------------------------
# 19. VIDEO-LEVEL QUALITY
# ------------------------------------------------------------

video_quality = (
    analysis_frames
    .groupby("uid")
    .agg(

        total_frames=(
            "uid",
            "size"
        ),

        saved_frames=(
            "roi_saved_bool",
            "sum"
        ),

        direct_detection_frames=(
            "is_direct_detection",
            "sum"
        ),

        temporal_fallback_frames=(
            "is_temporal_fallback",
            "sum"
        ),

        full_frame_fallback_frames=(
            "is_full_frame_fallback",
            "sum"
        ),

        one_hand_frames=(
            "is_one_hand",
            "sum"
        ),

        two_hand_frames=(
            "is_two_hands",
            "sum"
        ),

        mean_roi_area_fraction=(
            "bbox_area_fraction",
            "mean"
        ),

        median_roi_area_fraction=(
            "bbox_area_fraction",
            "median"
        ),

        mean_confidence_proxy=(
            "confidence_proxy",
            "mean"
        ),

        mean_latency_ms=(
            "latency_ms",
            "mean"
        ),

    )
    .reset_index()
)

video_quality["saved_fraction"] = (
    video_quality["saved_frames"]
    /
    video_quality["total_frames"]
)

video_quality["direct_detection_fraction"] = (
    video_quality["direct_detection_frames"]
    /
    video_quality["total_frames"]
)

video_quality["fallback_fraction"] = (
    (
        video_quality["temporal_fallback_frames"]
        +
        video_quality["full_frame_fallback_frames"]
    )
    /
    video_quality["total_frames"]
)

video_quality["one_hand_fraction"] = (
    video_quality["one_hand_frames"]
    /
    video_quality["total_frames"]
)

video_quality["two_hand_fraction"] = (
    video_quality["two_hand_frames"]
    /
    video_quality["total_frames"]
)

# ------------------------------------------------------------
# 20. VIDEO-LEVEL CONSISTENCY
# ------------------------------------------------------------

# A video is considered "direct-detection complete" when
# all 8 sampled frames have direct OpenCV detections.
#
# This is intentionally stricter than "ROI saved".
#
video_quality["all_8_direct"] = (
    video_quality["direct_detection_frames"]
    == 8
)

video_quality["at_least_6_direct"] = (
    video_quality["direct_detection_frames"]
    >= 6
)

video_quality["no_full_frame_fallback"] = (
    video_quality["full_frame_fallback_frames"]
    == 0
)

video_quality["no_temporal_fallback"] = (
    video_quality["temporal_fallback_frames"]
    == 0
)

video_quality["all_8_saved"] = (
    video_quality["saved_frames"]
    == 8
)

# ------------------------------------------------------------
# 21. TEMPORAL CONSISTENCY
# ------------------------------------------------------------
#
# We use the 8 prescribed sampled frames in temporal order.
#
# Two complementary measures are calculated:
#
#   A. ROI-area stability
#   B. ROI-center displacement
#
# The detector's bounding box is used for both.
#
# This avoids relying on raw video decoding again.
# ------------------------------------------------------------

analysis_frames = analysis_frames.sort_values(
    [
        "uid",
        "sample_number",
        "frame_index",
    ]
).copy()

analysis_frames["roi_center_x"] = (
    (
        analysis_frames["bbox_x1"]
        +
        analysis_frames["bbox_x2"]
    )
    / 2.0
)

analysis_frames["roi_center_y"] = (
    (
        analysis_frames["bbox_y1"]
        +
        analysis_frames["bbox_y2"]
    )
    / 2.0
)

analysis_frames["roi_center_norm_x"] = (
    analysis_frames["roi_center_x"]
    /
    analysis_frames["width"]
)

analysis_frames["roi_center_norm_y"] = (
    analysis_frames["roi_center_y"]
    /
    analysis_frames["height"]
)

temporal_records = []

for uid, group in analysis_frames.groupby("uid"):

    group = group.sort_values(
        [
            "sample_number",
            "frame_index",
        ]
    ).copy()

    # --------------------------------------------------------
    # ROI AREA TEMPORAL STABILITY
    # --------------------------------------------------------

    area_values = (
        group["bbox_area_fraction"]
        .dropna()
        .values
    )

    if len(area_values) >= 2:

        area_mean = np.mean(
            area_values
        )

        area_std = np.std(
            area_values
        )

        if area_mean > 0:

            area_cv = (
                area_std /
                area_mean
            )

            area_stability = (
                1.0 /
                (
                    1.0 +
                    area_cv
                )
            )

        else:

            area_cv = np.nan
            area_stability = 0.0

    else:

        area_cv = np.nan
        area_stability = 0.0

    # --------------------------------------------------------
    # CENTER DISPLACEMENT
    # --------------------------------------------------------

    centers = group[
        [
            "roi_center_norm_x",
            "roi_center_norm_y",
        ]
    ].dropna().values

    if len(centers) >= 2:

        displacement = np.sqrt(
            np.sum(
                np.diff(
                    centers,
                    axis=0
                ) ** 2,
                axis=1
            )
        )

        mean_center_displacement = float(
            np.mean(displacement)
        )

        median_center_displacement = float(
            np.median(displacement)
        )

        max_center_displacement = float(
            np.max(displacement)
        )

    else:

        mean_center_displacement = np.nan
        median_center_displacement = np.nan
        max_center_displacement = np.nan

    temporal_records.append({

        "uid":
            uid,

        "num_frames":
            len(group),

        "area_cv":
            area_cv,

        "area_stability":
            area_stability,

        "mean_center_displacement":
            mean_center_displacement,

        "median_center_displacement":
            median_center_displacement,

        "max_center_displacement":
            max_center_displacement,

    })

temporal_df = pd.DataFrame(
    temporal_records
)

# ------------------------------------------------------------
# 22. TEMPORAL QUALITY SUMMARY
# ------------------------------------------------------------

temporal_summary = {

    "videos_evaluated":
        int(len(temporal_df)),

    "mean_area_cv":
        float(
            temporal_df["area_cv"].mean()
        ),

    "median_area_cv":
        float(
            temporal_df["area_cv"].median()
        ),

    "mean_area_stability":
        float(
            temporal_df["area_stability"].mean()
        ),

    "median_area_stability":
        float(
            temporal_df["area_stability"].median()
        ),

    "mean_center_displacement":
        float(
            temporal_df[
                "mean_center_displacement"
            ].mean()
        ),

    "median_center_displacement":
        float(
            temporal_df[
                "median_center_displacement"
            ].median()
        ),

    "p95_center_displacement":
        float(
            temporal_df[
                "mean_center_displacement"
            ].quantile(0.95)
        ),

    "max_center_displacement":
        float(
            temporal_df[
                "max_center_displacement"
            ].max()
        ),
}

# ------------------------------------------------------------
# 23. MERGE TEMPORAL METRICS
# ------------------------------------------------------------

video_quality = video_quality.merge(
    temporal_df,
    on="uid",
    how="left"
)

# ------------------------------------------------------------
# 24. VIDEO-LEVEL QUALITY CATEGORIES
# ------------------------------------------------------------

video_quality["quality_category"] = "acceptable"

# Excellent:
# all 8 direct detections
video_quality.loc[
    video_quality["all_8_direct"],
    "quality_category"
] = "excellent"

# Good:
# at least 6 direct detections and
# no full-frame fallback
video_quality.loc[
    (
        video_quality["at_least_6_direct"]
        &
        video_quality["no_full_frame_fallback"]
        &
        ~video_quality["all_8_direct"]
    ),
    "quality_category"
] = "good"

# Mixed:
# saved but meaningful fallback usage
video_quality.loc[
    (
        (video_quality["saved_fraction"] == 1.0)
        &
        (~video_quality["at_least_6_direct"])
    ),
    "quality_category"
] = "mixed"

# ------------------------------------------------------------
# 25. ROI SIZE QUALITY CATEGORIES
# ------------------------------------------------------------

video_quality["roi_size_quality"] = "acceptable"

video_quality.loc[
    video_quality["mean_roi_area_fraction"] < ROI_MIN,
    "roi_size_quality"
] = "too_small"

video_quality.loc[
    video_quality["mean_roi_area_fraction"] > ROI_MAX,
    "roi_size_quality"
] = "too_large"

video_quality.loc[
    video_quality[
        "mean_roi_area_fraction"
    ].between(
        0.10,
        0.40,
        inclusive="both"
    ),
    "roi_size_quality"
] = "near_target"

# ------------------------------------------------------------
# 26. OVERALL VIDEO QUALITY SCORE
# ------------------------------------------------------------
#
# This is an audit score, NOT a model-training score.
#
# Components:
#
#   30% direct detection
#   20% saved ROI
#   15% one/two-hand evidence
#   15% ROI size
#   20% temporal stability
#
# The score is used only to summarize ROI quality.
# ------------------------------------------------------------

hand_evidence = (
    (
        video_quality["one_hand_fraction"]
        +
        video_quality["two_hand_fraction"]
    )
    .clip(
        0,
        1
    )
)

roi_size_score = (
    1.0
    -
    (
        (
            video_quality[
                "mean_roi_area_fraction"
            ]
            -
            ROI_TARGET
        )
        .abs()
        /
        ROI_TARGET
    )
).clip(
    0,
    1
)

temporal_score = (
    video_quality[
        "area_stability"
    ]
    .clip(
        0,
        1
    )
)

video_quality["quality_score"] = (

    0.30
    *
    video_quality[
        "direct_detection_fraction"
    ]

    +

    0.20
    *
    video_quality[
        "saved_fraction"
    ]

    +

    0.15
    *
    hand_evidence

    +

    0.15
    *
    roi_size_score

    +

    0.20
    *
    temporal_score
)

# ------------------------------------------------------------
# 27. QUALITY SCORE SUMMARY
# ------------------------------------------------------------

quality_score_summary = {

    "mean":
        float(
            video_quality[
                "quality_score"
            ].mean()
        ),

    "median":
        float(
            video_quality[
                "quality_score"
            ].median()
        ),

    "std":
        float(
            video_quality[
                "quality_score"
            ].std()
        ),

    "p05":
        float(
            video_quality[
                "quality_score"
            ].quantile(0.05)
        ),

    "p25":
        float(
            video_quality[
                "quality_score"
            ].quantile(0.25)
        ),

    "p75":
        float(
            video_quality[
                "quality_score"
            ].quantile(0.75)
        ),

    "p95":
        float(
            video_quality[
                "quality_score"
            ].quantile(0.95)
        ),
}

# ------------------------------------------------------------
# 28. QUALITY ACCEPTANCE THRESHOLDS
# ------------------------------------------------------------
#
# These thresholds are used for audit reporting.
#
# We DO NOT remove videos based on these thresholds.
#
# The 6,628-video frozen population remains fixed.
# ------------------------------------------------------------

DIRECT_DETECTION_ACCEPTABLE = 0.50
FALLBACK_ACCEPTABLE = 0.50
ROI_SIZE_ACCEPTABLE_MIN = 0.005
ROI_SIZE_ACCEPTABLE_MAX = 0.90
TEMPORAL_STABILITY_ACCEPTABLE = 0.40

overall_direct_rate = frame_summary[
    "direct_detection_rate"
]

overall_fallback_rate = frame_summary[
    "any_fallback_rate"
]

overall_roi_valid_rate = frame_summary[
    "roi_area_valid_rate"
]

overall_temporal_stability = temporal_summary[
    "mean_area_stability"
]

quality_checks = {

    "direct_detection_rate_above_threshold":
        overall_direct_rate
        >= DIRECT_DETECTION_ACCEPTABLE,

    "fallback_rate_below_threshold":
        overall_fallback_rate
        <= FALLBACK_ACCEPTABLE,

    "roi_area_valid_rate_above_threshold":
        overall_roi_valid_rate
        >= 0.95,

    "temporal_stability_above_threshold":
        overall_temporal_stability
        >= TEMPORAL_STABILITY_ACCEPTABLE,

}

# ------------------------------------------------------------
# 29. IMPORTANT: NO DATA EXCLUSION HERE
# ------------------------------------------------------------
#
# Stage 5.4 validates quality.
#
# It does NOT create a smaller subset.
#
# The 6,628 UID set remains frozen.
# ------------------------------------------------------------

quality_decision = (
    "PASS"
    if all(quality_checks.values())
    else "PASS_WITH_REVIEW"
)

# ------------------------------------------------------------
# 30. SAVE FRAME-LEVEL QUALITY AUDIT
# ------------------------------------------------------------

frame_quality_path = (
    OUTPUT_DIR /
    "stage5_4_frame_level_quality_audit.csv"
)

analysis_frames.to_csv(
    frame_quality_path,
    index=False
)

# ------------------------------------------------------------
# 31. SAVE VIDEO-LEVEL QUALITY AUDIT
# ------------------------------------------------------------

video_quality_path = (
    OUTPUT_DIR /
    "stage5_4_video_level_quality_audit.csv"
)

video_quality.to_csv(
    video_quality_path,
    index=False
)

# ------------------------------------------------------------
# 32. SAVE TEMPORAL QUALITY AUDIT
# ------------------------------------------------------------

temporal_path = (
    OUTPUT_DIR /
    "stage5_4_temporal_consistency_audit.csv"
)

temporal_df.to_csv(
    temporal_path,
    index=False
)

# ------------------------------------------------------------
# 33. SAVE FALLBACK DISTRIBUTION
# ------------------------------------------------------------

fallback_path = (
    OUTPUT_DIR /
    "stage5_4_fallback_distribution.csv"
)

fallback_distribution.to_csv(
    fallback_path,
    index=False
)

# ------------------------------------------------------------
# 34. SAVE HAND COUNT DISTRIBUTION
# ------------------------------------------------------------

hand_distribution_path = (
    OUTPUT_DIR /
    "stage5_4_hand_count_distribution.csv"
)

hand_distribution.to_csv(
    hand_distribution_path,
    index=False
)

# ------------------------------------------------------------
# 35. SAVE SUMMARY JSON
# ------------------------------------------------------------

summary = {

    "stage":
        "5.4",

    "experiment":
        "ROI Quality Validation",

    "timestamp":
        datetime.now().isoformat(),

    "analysis_videos":
        int(analysis_videos["uid"].nunique()),

    "analysis_frames":
        int(len(analysis_frames)),

    "frames_per_video":
        8,

    "detector":
        "OpenCV_Skin_Contour",

    "backbone":
        "MobileNetV3-Large",

    "frame_level_summary":
        frame_summary,

    "confidence_summary":
        confidence_summary,

    "roi_area_summary":
        roi_area_summary,

    "latency_summary":
        latency_summary,

    "temporal_summary":
        temporal_summary,

    "quality_score_summary":
        quality_score_summary,

    "quality_checks":
        quality_checks,

    "overall_quality_decision":
        quality_decision,

    "important_note":
        (
            "Stage 5.4 is a validation-only stage. "
            "No videos were removed from the frozen "
            "analysis-valid population based on ROI quality."
        ),

    "raw_videos_modified":
        False,

    "stage5_3_outputs_modified":
        False,

    "stage4_modified":
        False,

}

summary_path = (
    OUTPUT_DIR /
    "stage5_4_roi_quality_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 36. PLOT 1 — FALLBACK DISTRIBUTION
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

fallback_plot = fallback_distribution.copy()

plt.bar(
    fallback_plot["fallback_type"].astype(str),
    fallback_plot["fraction"]
)

plt.ylabel(
    "Fraction of sampled frames"
)

plt.xlabel(
    "Extraction outcome"
)

plt.title(
    "Stage 5.4 — ROI Extraction Outcome Distribution"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    PLOTS_DIR /
    "stage5_4_fallback_distribution.png",
    dpi=200
)

plt.close()

# ------------------------------------------------------------
# 37. PLOT 2 — ROI AREA DISTRIBUTION
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.hist(
    roi_area.dropna(),
    bins=50
)

plt.axvline(
    ROI_TARGET,
    linestyle="--",
    label="20% target"
)

plt.xlabel(
    "ROI area fraction"
)

plt.ylabel(
    "Frame count"
)

plt.title(
    "Stage 5.4 — Hand ROI Area Distribution"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    PLOTS_DIR /
    "stage5_4_roi_area_distribution.png",
    dpi=200
)

plt.close()

# ------------------------------------------------------------
# 38. PLOT 3 — CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.hist(
    confidence.dropna(),
    bins=50
)

plt.xlabel(
    "OpenCV confidence proxy"
)

plt.ylabel(
    "Frame count"
)

plt.title(
    "Stage 5.4 — OpenCV ROI Confidence Proxy"
)

plt.tight_layout()

plt.savefig(
    PLOTS_DIR /
    "stage5_4_confidence_distribution.png",
    dpi=200
)

plt.close()

# ------------------------------------------------------------
# 39. PLOT 4 — TEMPORAL STABILITY
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.hist(
    temporal_df[
        "area_stability"
    ].dropna(),
    bins=40
)

plt.xlabel(
    "ROI area temporal stability"
)

plt.ylabel(
    "Video count"
)

plt.title(
    "Stage 5.4 — Temporal ROI Stability"
)

plt.tight_layout()

plt.savefig(
    PLOTS_DIR /
    "stage5_4_temporal_stability.png",
    dpi=200
)

plt.close()

# ------------------------------------------------------------
# 40. PLOT 5 — VIDEO QUALITY SCORE
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.hist(
    video_quality[
        "quality_score"
    ].dropna(),
    bins=40
)

plt.xlabel(
    "ROI quality score"
)

plt.ylabel(
    "Video count"
)

plt.title(
    "Stage 5.4 — Video-Level ROI Quality Score"
)

plt.tight_layout()

plt.savefig(
    PLOTS_DIR /
    "stage5_4_quality_score_distribution.png",
    dpi=200
)

plt.close()

# ------------------------------------------------------------
# 41. SAVE HUMAN-READABLE REPORT
# ------------------------------------------------------------

report_lines = []

report_lines.append(
    "STAGE 5.4 — ROI QUALITY VALIDATION"
)

report_lines.append(
    "=" * 80
)

report_lines.append(
    f"Analysis-valid videos: "
    f"{analysis_videos['uid'].nunique()}"
)

report_lines.append(
    f"Analysis frames: "
    f"{len(analysis_frames)}"
)

report_lines.append(
    f"Frames per video: 8"
)

report_lines.append(
    ""
)

report_lines.append(
    "FRAME-LEVEL QUALITY"
)

report_lines.append(
    f"Frame read success rate: "
    f"{frame_summary['frame_read_success_rate']:.4f}"
)

report_lines.append(
    f"ROI saved rate: "
    f"{frame_summary['roi_saved_rate']:.4f}"
)

report_lines.append(
    f"Direct OpenCV detection rate: "
    f"{frame_summary['direct_detection_rate']:.4f}"
)

report_lines.append(
    f"Temporal fallback rate: "
    f"{frame_summary['temporal_fallback_rate']:.4f}"
)

report_lines.append(
    f"Full-frame fallback rate: "
    f"{frame_summary['full_frame_fallback_rate']:.4f}"
)

report_lines.append(
    f"Any fallback rate: "
    f"{frame_summary['any_fallback_rate']:.4f}"
)

report_lines.append(
    f"One-hand rate: "
    f"{frame_summary['one_hand_rate']:.4f}"
)

report_lines.append(
    f"Two-hand rate: "
    f"{frame_summary['two_hand_rate']:.4f}"
)

report_lines.append(
    f"Zero-hand rate: "
    f"{frame_summary['zero_hand_rate']:.4f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "ROI AREA"
)

report_lines.append(
    f"Mean ROI area fraction: "
    f"{roi_area_summary['mean']:.6f}"
)

report_lines.append(
    f"Median ROI area fraction: "
    f"{roi_area_summary['median']:.6f}"
)

report_lines.append(
    f"P05 ROI area fraction: "
    f"{roi_area_summary['p05']:.6f}"
)

report_lines.append(
    f"P95 ROI area fraction: "
    f"{roi_area_summary['p95']:.6f}"
)

report_lines.append(
    f"ROI-valid rate: "
    f"{frame_summary['roi_area_valid_rate']:.4f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "CONFIDENCE"
)

report_lines.append(
    f"Mean confidence proxy: "
    f"{confidence_summary['mean']:.6f}"
)

report_lines.append(
    f"Median confidence proxy: "
    f"{confidence_summary['median']:.6f}"
)

report_lines.append(
    f"P05 confidence proxy: "
    f"{confidence_summary['p05']:.6f}"
)

report_lines.append(
    f"P95 confidence proxy: "
    f"{confidence_summary['p95']:.6f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "TEMPORAL CONSISTENCY"
)

report_lines.append(
    f"Mean area stability: "
    f"{temporal_summary['mean_area_stability']:.6f}"
)

report_lines.append(
    f"Median area stability: "
    f"{temporal_summary['median_area_stability']:.6f}"
)

report_lines.append(
    f"Mean center displacement: "
    f"{temporal_summary['mean_center_displacement']:.6f}"
)

report_lines.append(
    f"P95 center displacement: "
    f"{temporal_summary['p95_center_displacement']:.6f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "VIDEO-LEVEL QUALITY"
)

report_lines.append(
    f"All 8 frames directly detected: "
    f"{video_quality['all_8_direct'].mean():.4f}"
)

report_lines.append(
    f"At least 6/8 directly detected: "
    f"{video_quality['at_least_6_direct'].mean():.4f}"
)

report_lines.append(
    f"No full-frame fallback: "
    f"{video_quality['no_full_frame_fallback'].mean():.4f}"
)

report_lines.append(
    f"All 8 ROI images saved: "
    f"{video_quality['all_8_saved'].mean():.4f}"
)

report_lines.append(
    f"Mean video quality score: "
    f"{quality_score_summary['mean']:.6f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "OVERALL QUALITY DECISION"
)

report_lines.append(
    quality_decision
)

report_lines.append(
    ""
)

report_lines.append(
    "IMPORTANT"
)

report_lines.append(
    "No videos were removed based on ROI quality."
)

report_lines.append(
    "The frozen 6,628-video analysis population remains unchanged."
)

report_path = (
    OUTPUT_DIR /
    "stage5_4_human_readable_report.txt"
)

with open(
    report_path,
    "w"
) as f:

    f.write(
        "\n".join(report_lines)
    )

# ------------------------------------------------------------
# 42. FINAL CONSOLE OUTPUT
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("STAGE 5.4 — ROI QUALITY VALIDATION COMPLETE")
print("=" * 100)

print(
    f"\nAnalysis-valid videos: "
    f"{analysis_videos['uid'].nunique()}"
)

print(
    f"Analysis frames: "
    f"{len(analysis_frames)}"
)

print("\n--- FRAME-LEVEL QUALITY ---")

print(
    f"Frame read success:       "
    f"{frame_summary['frame_read_success_rate']:.4f}"
)

print(
    f"ROI saved:                "
    f"{frame_summary['roi_saved_rate']:.4f}"
)

print(
    f"Direct OpenCV detection:  "
    f"{frame_summary['direct_detection_rate']:.4f}"
)

print(
    f"Temporal fallback:        "
    f"{frame_summary['temporal_fallback_rate']:.4f}"
)

print(
    f"Full-frame fallback:      "
    f"{frame_summary['full_frame_fallback_rate']:.4f}"
)

print(
    f"Any fallback:             "
    f"{frame_summary['any_fallback_rate']:.4f}"
)

print(
    f"One-hand:                 "
    f"{frame_summary['one_hand_rate']:.4f}"
)

print(
    f"Two-hand:                 "
    f"{frame_summary['two_hand_rate']:.4f}"
)

print("\n--- ROI AREA ---")

print(
    f"Mean ROI area:            "
    f"{roi_area_summary['mean']:.6f}"
)

print(
    f"Median ROI area:          "
    f"{roi_area_summary['median']:.6f}"
)

print(
    f"ROI-valid rate:           "
    f"{frame_summary['roi_area_valid_rate']:.4f}"
)

print("\n--- CONFIDENCE ---")

print(
    f"Mean confidence proxy:    "
    f"{confidence_summary['mean']:.6f}"
)

print(
    f"Median confidence proxy:  "
    f"{confidence_summary['median']:.6f}"
)

print("\n--- TEMPORAL CONSISTENCY ---")

print(
    f"Mean area stability:      "
    f"{temporal_summary['mean_area_stability']:.6f}"
)

print(
    f"Median area stability:    "
    f"{temporal_summary['median_area_stability']:.6f}"
)

print(
    f"Mean center displacement: "
    f"{temporal_summary['mean_center_displacement']:.6f}"
)

print("\n--- VIDEO-LEVEL QUALITY ---")

print(
    f"All 8 direct detections:  "
    f"{video_quality['all_8_direct'].mean():.4f}"
)

print(
    f"At least 6/8 direct:      "
    f"{video_quality['at_least_6_direct'].mean():.4f}"
)

print(
    f"No full-frame fallback:   "
    f"{video_quality['no_full_frame_fallback'].mean():.4f}"
)

print(
    f"All 8 ROI saved:          "
    f"{video_quality['all_8_saved'].mean():.4f}"
)

print(
    f"Mean quality score:       "
    f"{quality_score_summary['mean']:.6f}"
)

print("\n--- FINAL DECISION ---")

print(
    f"ROI quality decision:     {quality_decision}"
)

print("\nArtifacts saved to:")

print(
    OUTPUT_DIR
)

print("\nGenerated files:")

for p in sorted(OUTPUT_DIR.rglob("*")):

    if p.is_file():

        print(
            "  ",
            p.relative_to(OUTPUT_DIR)
        )

print("\n" + "=" * 100)

print("""
STAGE 5.4 IS VALIDATION-ONLY.

No raw videos were modified.
No ROI images were regenerated.
No Stage 5.3 extraction was rerun.
No Stage 4 artifacts were modified.
No videos were removed from the frozen 6,628-video population.

The same 6,628 UIDs remain frozen for Stage 5.5–5.7.
""")

print("=" * 100)

STAGE 5.4 — ROI QUALITY VALIDATION

Input verification:

✓ Stage 5.3 frame audit
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_frame_level_extraction.csv
✓ Stage 5.3 video audit
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_video_level_summary.csv
✓ Frozen analysis-valid UID manifest
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/analysis_valid_subset/stage5_3_analysis_valid_uid_manifest.csv

All Stage 5.4 inputs found.

Loaded shapes:
Frame audit : (56392, 30)
Video audit : (7049, 23)
Valid UID manifest : (6628, 4)

Frozen analysis UID count:
6628

Matched analysis population:
Videos : 6628
Frames : 53024

✓ Confirmed 6628 videos × 8 frames = 53024 frame records.

✓ Required frame-level quality columns present.


STAGE 5.4 — ROI QUALITY VALIDATION COMPLETE

Analysis-valid videos: 6628
Analysis frames: 53024

--- FRAME-LEVEL QUALITY ---
Frame read success:       1.0000
ROI save